# Hybrid: 22.98 payload's invsqrt+dqi with the newer v44 chemistry
`yoyobar/qcc-2298-submission-20261030` supplies invsqrt and dqi;
`yoyobar/qcc-chem-v44-20261030` supplies a newer 15-circuit chemistry set with its own npz.


In [ ]:

import glob, pathlib, zipfile

# Hybrid: invsqrt + dqi from the 22.98 payload, chemistry from the newer v44 set.
payload = [str(pathlib.Path(p).parent) for p in glob.glob('/kaggle/input/**/submission/invsqrt.km*', recursive=True)
           if '/competitions/' not in p.replace('\\', '/')]
chem = [str(pathlib.Path(p).parent) for p in glob.glob('/kaggle/input/**/submission.npz', recursive=True)
        if '/competitions/' not in p.replace('\\', '/') and 'v44' in p.lower()]
print('payload:', payload)
print('chem v44:', chem)
assert payload and chem
p_root = pathlib.Path(payload[0]); c_root = pathlib.Path(chem[0])

dst = pathlib.Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zo:
    zo.write(p_root / 'invsqrt.kmb', 'invsqrt.kmb')
    zo.write(p_root / 'dqi.kmb', 'dqi.kmb')
    circuits = sorted(c_root.glob('angle_k_*.kmb'), key=lambda p: int(p.stem.rsplit('_', 1)[1]))
    for p in circuits:
        zo.write(p, 'dfthcsos/' + p.name)
    zo.write(c_root / 'submission.npz', 'dfthcsos/submission.npz')

print('wrote', dst, dst.stat().st_size)
with zipfile.ZipFile(dst) as z:
    names = z.namelist()
circ = [n for n in names if n.startswith('dfthcsos/') and n.endswith(('.kmb', '.kmx'))]
print('entries', len(names), '| circuits', len(circ))
assert len(circ) == len(circuits)
assert sum(1 for n in names if n.startswith('invsqrt.')) == 1
assert sum(1 for n in names if n.startswith('dqi.')) == 1
print('layout OK')
